## INIT

In [ ]:
from pathlib import Path
import sys

# Locate lego/scripts and derive lego root
cwd = Path.cwd()
lego_root = None
for base in [cwd] + list(cwd.parents):
    direct = base / 'scripts'
    nested = base / 'lego' / 'scripts'
    if direct.exists():
        lego_root = direct.parent
        break
    if nested.exists():
        lego_root = nested.parent
        break
if lego_root is None:
    raise FileNotFoundError('lego/scripts directory not found')
sys.path.insert(0, str(lego_root.parent))

# Convenience paths
SCRIPTS_DIR = lego_root / 'scripts'
NOTEBOOKS_DIR = lego_root / 'notebooks'


In [ ]:
import pandas as pd 
import duckdb
import sys

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(repo_root))
from lego.scripts.inventory import load_my_sets_from_csv, build_my_inventory


In [ ]:
con = duckdb.connect("/Users/alialvarez/Documents/databases/lego.duckdb")
print(con.execute("PRAGMA database_list").fetchall())


In [ ]:
#refresh data
from pathlib import Path
from lego.scripts.inventory import load_my_sets_from_csv, build_my_inventory

load_my_sets_from_csv(con, Path("/Volumes/Datasets/lego/my_sets.csv"))

In [ ]:
from pathlib import Path
from lego.scripts.inventory import load_my_sets_from_csv

con.execute("DROP TABLE IF EXISTS my_sets")
load_my_sets_from_csv(con, Path("/Volumes/Datasets/lego/my_sets.csv"))

print(con.execute("DESCRIBE my_sets").fetchall())


### Functions

In [ ]:
def query(q, con, df_return=False):
    df = con.execute(q).df()
    display(df)
    if df_return:
        return df


## Building coverage

In [ ]:
import importlib
import lego.scripts.inventory as inv
importlib.reload(inv)

from lego.scripts.inventory import build_model_coverage, build_candidate_models

build_model_coverage(con)
build_candidate_models(con)

con.execute("CREATE TABLE IF NOT EXISTS favorite_keywords(keyword TEXT)")


In [ ]:
import lego.scripts.inventory as inv
print(inv.__file__)
import inspect
print(inspect.getsource(inv.build_candidate_models))


In [ ]:
con.execute("CREATE TABLE IF NOT EXISTS favorite_keywords(keyword TEXT)")


## Exploring Tables

In [ ]:
# 1) How many models meet coverage threshold?
q='''
SELECT 
s.set_num, s.name, total_required, total_owned, coverage_pct
FROM model_coverage
LEFT JOIN sets s using (set_num)
LEFT JOIN my_sets m on s.set_num = m.set_num
WHERE coverage_pct >= 0.85
-- is not owned
AND m.set_num IS NULL
-- filter out smaller models
AND total_required > 10
ORDER BY coverage_pct DESC
'''
df=query(q,con,True)



In [ ]:
# 2) How many models match keywords by theme or set name?
q='''
SELECT COUNT(*)
FROM model_coverage mc
JOIN sets s
  ON TRY_CAST(SPLIT_PART(CAST(s.set_num AS TEXT), '-', 1) AS INTEGER) = mc.model_id
LEFT JOIN themes t ON t.id = s.theme_id
LEFT JOIN favorite_keywords fk
  ON LOWER(fk.keyword) = LOWER(t.name)
  OR LOWER(fk.keyword) = LOWER(s.name)
WHERE fk.keyword IS NOT NULL
'''
df=query(q,con,True)

In [ ]:
q='''
SELECT * FROM candidate_models;
'''
df=query(q,con,True)

In [ ]:
q='''
SELECT * FROM SETS s
LEFT JOIN themes t ON s.theme_id=t.id 
'''
df=query(q,con,True)

In [ ]:
q='''
SHOW TABLES
'''
df=query(q,con,True)

In [ ]:
for table in df.name.to_list():
    print(table,"\n")
    q=f'''
    SELECT *
    FROM {table}
    LIMIT 20;
    '''
    query(q,con,False)